In [48]:
import json
import pandas as pd

# HLTM
topic_map_filepath = './HLTM/output-jsons/' + 'T3.topics.json'

with open(topic_map_filepath, 'r') as f:
    data = json.load(f)

topic_data = []
for entry in data:
    topic_id = entry['topic']
    video_ids = [doc[0] for doc in entry['doc']]  # discard probabilities
    topic_data.append({'topic_id': topic_id, 'video_ids': video_ids})

# Convert to DataFrame
hltm_df = pd.DataFrame(topic_data)

# Display the first few rows
hltm_df.head()

hltm_labels = pd.read_csv('./HLTM/!POSTRUN/' + 'T3_categories.csv')


hltm_df = hltm_df.merge(hltm_labels, left_on='topic_id', right_on='Id', how='left')  

hltm_df = hltm_df[['topic_id', 'video_ids', 'Specific Category', 'General Category']]

hltm_df.head()

,topic_id,video_ids,Specific Category,General Category
0,Z22,"[65, 105, 321, 337, 494, 514, 536, 721, 759, 8...",Viral Latino Coach and Podcast Features,Entertainment
1,Z1174,"[5, 19, 21, 25, 41, 43, 52, 54, 71, 76, 86, 88...",Emotional Resilience,Others
2,Z1197,"[1, 3, 8, 17, 27, 33, 41, 43, 51, 52, 54, 58, ...",Business Issues,Lifestyle
3,Z11,"[811, 2479, 538, 1908, 2643, 887, 1869, 2280, ...",Concert and Studio Music Performances,Entertainment
4,Z232,"[51, 144, 168, 191, 343, 660, 685, 732, 1054, ...",Taxi Rides and Flights,Travel


In [49]:
categories = ['Food', 'Entertainment', 'Lifestyle', 'Others', 'Travel', 'Culture', 'Politics']
categories_df = pd.DataFrame({
    'Category': categories,
    'Video_IDs': [set() for _ in categories]  # Initialize with empty sets
})
categories_df.head()

hltm_df['video_ids'] = hltm_df['video_ids'].apply(lambda x: list(map(str, x)))



In [50]:
print("Unique General Categories in data:", hltm_df['General Category'].unique())

for _, row in hltm_df.iterrows():
    general_category = row['General Category']
    if general_category in categories:
        idx = categories_df[categories_df['Category'] == general_category].index[0]
        categories_df.at[idx, 'Video_IDs'].update(row['video_ids'])

# Optional: Convert sets to sorted lists for viewing or export
categories_df['Video_IDs'] = categories_df['Video_IDs'].apply(lambda s: sorted(list(s)))

# Convert all video IDs to integers and sort
categories_df['Video_IDs'] = categories_df['Video_IDs'].apply(lambda ids: sorted([int(x) for x in ids]))

# Display the resultq
print(categories_df)

categories_df['Video_Count'] = categories_df['Video_IDs'].apply(len)
print(categories_df[['Category', 'Video_Count']])

Unique General Categories in data: ['Entertainment' 'Others' 'Lifestyle' 'Travel' 'Food' 'Culture' 'Politics']
        Category                                          Video_IDs
0           Food  [0, 2, 4, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17,...
1  Entertainment  [0, 1, 3, 4, 5, 6, 7, 9, 11, 12, 17, 18, 19, 2...
2      Lifestyle  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
3         Others  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
4         Travel  [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 17...
5        Culture  [1, 4, 5, 6, 7, 8, 9, 10, 13, 15, 18, 19, 21, ...
6       Politics  [0, 1, 4, 7, 8, 9, 10, 11, 18, 19, 20, 21, 23,...
        Category  Video_Count
0           Food         2110
1  Entertainment         2113
2      Lifestyle         2550
3         Others         2573
4         Travel         2213
5        Culture         1663
6       Politics         1479
